# Bottom Ring Hole Pattern Classifier — v1

EfficientNetB0, trained on bottom-ring hole-pattern crops.

- **Classes:** bharat, hp, indane (3 classes)
- **Architecture:** EfficientNetB0 — lighter than the B2 used for the main brand classifier, since the bottom-ring hole pattern is a simpler, more localized visual signal that doesn't need the extra capacity
- **Augmentation:** ColorJitter, horizontal flip, full `RandomRotation(180)` — the ring can appear at any orientation depending on how the cylinder is set down, so rotation isn't limited to a small angle range like the main classifier
- **Output:** `classifier_bottomring_v1.pth`, saved to Drive under `LPG Cylinder Detection and Classification/Classifier/bottomring_v1/`

## How to Run
1. **Runtime:** Colab GPU (any tier — this is a small EfficientNetB0 run on a modest dataset).
2. **Mount Drive** (cell 1) — used both to optionally source the dataset zip and to push the
   final checkpoint back to Drive.
3. **Provide the dataset** (cell 2): either upload a zip directly (`UPLOAD_LOCAL = True`, the
   default) or set `UPLOAD_LOCAL = False` and point `DRIVE_ZIP_PATH` at a zip already in Drive.
   The zip must extract to `bottomring/train/{bharat,hp,indane}` and
   `bottomring/valid/{bharat,hp,indane}`.
4. **Execution order:** run top to bottom — imports, transforms, and dataset/sampler cells must
   run before the model/training cells, which in turn must finish before evaluation and the
   save/push-to-Drive cells at the end.
5. **Expected outputs:** `classifier_bottomring_v1.pth` checkpoint, a training-history JSON, and
   both files pushed to the Drive folder above (see Output Files table near the end).

> **Note (v2 gap):** the model actually shipped in `models/classifier_bottomring_v2.pth` is a
> *later* version whose training code is not present in this repo (see `CLAUDE.md` — "Known
> Issues"). This notebook is the v1 lineage only; do not assume it reproduces v2.

## Model / Dataset Info
| | |
|---|---|
| Architecture | EfficientNetB0 + `Dropout(0.3)` head |
| Dataset | `bottomring/{train,valid}/{bharat,hp,indane}` (counts printed at runtime in cell 2 — not hardcoded in this notebook) |
| Classes | `bharat`, `hp`, `indane` |
| Loss / Optim | `CrossEntropyLoss`, `AdamW` (lr=3e-4, weight_decay=1e-4), `CosineAnnealingLR(T_max=30)` |
| Sampling | `WeightedRandomSampler` to counter class imbalance |
| Epochs | 30 |
| Best val acc | printed at runtime (`best_val_acc`) — see training loop / eval cell output |

## Current Status
Complete and self-contained: mount → data → train → evaluate → save/push to Drive. No TODOs or
known-broken cells in this notebook itself. The known gap is external to this file — this is the
v1 checkpoint lineage, and the v2 checkpoint currently in production (`classifier_bottomring_v2.pth`)
has no corresponding training notebook in the repo.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Upload / unzip dataset

Expects a zip that extracts to `bottomring/train/{bharat,hp,indane}` and `bottomring/valid/{bharat,hp,indane}`.

Either upload the zip directly, or point `DRIVE_ZIP_PATH` at a zip already sitting in Drive.

In [ ]:
import os

os.makedirs('/content/dataset', exist_ok=True)

# Option A — upload a zip from local disk
UPLOAD_LOCAL = True

# Option B — copy a zip already in Drive (set UPLOAD_LOCAL = False to use this)
DRIVE_ZIP_PATH = '/content/drive/MyDrive/LPG Cylinder Detection and Classification/Dataset/bottomring.zip'  # ← UPDATE THIS PATH

if UPLOAD_LOCAL:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
else:
    import shutil
    zip_name = 'bottomring.zip'
    shutil.copy(DRIVE_ZIP_PATH, f'/content/{zip_name}')

print(f'Using zip: {zip_name}')

In [ ]:
import zipfile

with zipfile.ZipFile(f'/content/{zip_name}', 'r') as zf:
    zf.extractall('/content/dataset/bottomring')

TRAIN_DIR = '/content/dataset/bottomring/train'
VALID_DIR = '/content/dataset/bottomring/valid'
CLASSES   = ['bharat', 'hp', 'indane']

for split_dir in (TRAIN_DIR, VALID_DIR):
    print(split_dir)
    for c in CLASSES:
        p = os.path.join(split_dir, c)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f'  {c}: {n}')

## 3. Imports

In [ ]:
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms as T

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 4. Transforms

`RandomRotation(180)` (full rotation) since the bottom ring's hole pattern can be photographed from any orientation — unlike the main brand classifier, there's no consistent "upright" reference.

In [ ]:
IMG_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.2),
])

valid_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 5. Dataset + weighted sampler

`WeightedRandomSampler` compensates for class imbalance across bharat/hp/indane.

In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=valid_transform)

assert train_dataset.classes == CLASSES, f'Class order mismatch: {train_dataset.classes} vs {CLASSES}'
assert valid_dataset.classes == CLASSES, f'Class order mismatch: {valid_dataset.classes} vs {CLASSES}'

class_counts = np.bincount(train_dataset.targets, minlength=len(CLASSES))
print('Train class counts:', dict(zip(CLASSES, class_counts)))

class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_dataset.targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} images | Valid: {len(valid_dataset)} images')

## 6. Model — EfficientNetB0

Plain EfficientNetB0 classifier head — no spatial attention. The bottom-ring hole pattern is a simple, localized shape signal, so the lighter B0 backbone with a standard dropout+linear head is enough capacity without the overfitting risk of a bigger model on a smaller/simpler dataset.

In [ ]:
def build_model(num_classes=3):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model


model = build_model(num_classes=len(CLASSES)).to(device)
print(model.__class__.__name__, '- params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Training setup

In [ ]:
EPOCHS = 30
LR = 3e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. Training loop

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    return total_loss / total, 100.0 * correct / total

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_state = None

start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(valid_loader, train=False)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    marker = ' *' if is_best else ''
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss {train_loss:.4f} train_acc {train_acc:.2f}% | '
          f'val_loss {val_loss:.4f} val_acc {val_acc:.2f}%{marker}')

elapsed = time.time() - start
print(f'\nTraining done in {elapsed/60:.1f} min | best val acc: {best_val_acc:.2f}%')

model.load_state_dict(best_state)

## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Evaluation + confusion matrix

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Bottom Ring Classifier v1 — val acc {best_val_acc:.2f}%')
plt.tight_layout()
plt.show()

## 11. Save checkpoint + push to Drive

In [ ]:
import json

CHECKPOINT_NAME = 'classifier_bottomring_v1.pth'
LOCAL_CKPT_PATH = f'/content/{CHECKPOINT_NAME}'

checkpoint = {
    'model_state_dict':     model.state_dict(),
    'best_val_acc':         best_val_acc,
    'classes':              CLASSES,
    'architecture':         'efficientnet_b0',
    'confidence_threshold': 0.60,
    'epochs':               EPOCHS,
    'batch_size':            BATCH_SIZE,
    'lr':                    LR,
    'rotation_degrees':      180,
    'img_size':              IMG_SIZE,
    'notes':                 'Bottom ring hole-pattern classifier; full 180-degree rotation augmentation since ring orientation is unconstrained',
}

torch.save(checkpoint, LOCAL_CKPT_PATH)
print(f'Saved checkpoint locally: {LOCAL_CKPT_PATH}')

with open('/content/training_history_bottomring_v1.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/LPG Cylinder Detection and Classification/Classifier/bottomring_v1'  # ← UPDATE THIS PATH
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(LOCAL_CKPT_PATH, os.path.join(DRIVE_SAVE_DIR, CHECKPOINT_NAME))
shutil.copy('/content/training_history_bottomring_v1.json', os.path.join(DRIVE_SAVE_DIR, 'training_history_bottomring_v1.json'))

print(f'Saved to Drive: {DRIVE_SAVE_DIR}')
print(f'  - {CHECKPOINT_NAME}')
print(f'  - training_history_bottomring_v1.json')
print(f'\nBest val acc: {best_val_acc:.2f}%')

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `classifier_bottomring_v1.pth` | `/content/`, then copied to `DRIVE_SAVE_DIR` | Checkpoint dict: `model_state_dict`, `best_val_acc`, `classes`, `architecture`, `confidence_threshold`, `epochs`, `batch_size`, `lr`, `rotation_degrees`, `img_size`, `notes` |
| `training_history_bottomring_v1.json` | `/content/`, then copied to `DRIVE_SAVE_DIR` | Per-epoch `train_loss`/`train_acc`/`val_loss`/`val_acc` history dict |

`DRIVE_SAVE_DIR` = `LPG Cylinder Detection and Classification/Classifier/bottomring_v1/` (see
`# ← UPDATE THIS PATH` comment above).